In [3]:
import os
import cv2
import numpy as np
from glob import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score

# ==========================================
# 1. EARLY STOPPING CLASS
# ==========================================
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

# ==========================================
# 2. DATASET (Supports JPG and PNG)
# ==========================================
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, augment=False):
        extensions = ['*.png', '*.jpg', '*.jpeg']
        self.image_paths = []
        self.mask_paths = []
        
        for ext in extensions:
            self.image_paths.extend(glob(os.path.join(image_dir, ext)))
            self.mask_paths.extend(glob(os.path.join(mask_dir, ext)))
            
        self.image_paths = sorted(self.image_paths)
        self.mask_paths = sorted(self.mask_paths)
        self.augment = augment
        
        print(f"Dataset Initialized: Found {len(self.image_paths)} images/masks.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        
        image = cv2.resize(image, (256, 256))
        mask = cv2.resize(mask, (256, 256))

        if self.augment:
            if np.random.random() > 0.5:
                image = np.flip(image, axis=1)
                mask = np.flip(mask, axis=1)

        image = image.transpose((2, 0, 1)).astype(np.float32) / 255.0
        mask = (mask > 128).astype(np.float32)
        mask = mask[np.newaxis, ...]

        return torch.tensor(image.copy()), torch.tensor(mask.copy())

# ==========================================
# 3. LOSS AND MODEL
# ==========================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    def forward(self, predict, target):
        predict = predict.view(-1)
        target = target.view(-1)
        intersection = (predict * target).sum()
        dice = (2. * intersection + self.smooth) / (predict.sum() + target.sum() + self.smooth)
        return 1 - dice

class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        def CBR(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU(inplace=True)
            )
        self.enc1 = CBR(3, 64); self.enc2 = CBR(64, 128); self.enc3 = CBR(128, 256); self.enc4 = CBR(256, 512)
        self.pool = nn.MaxPool2d(2); self.bottleneck = CBR(512, 1024)
        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2); self.dec4 = CBR(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2); self.dec3 = CBR(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2); self.dec2 = CBR(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2); self.dec1 = CBR(128, 64)
        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.upconv4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.upconv3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.upconv1(d2), e1], dim=1))
        return torch.sigmoid(self.final(d1))

# ==========================================
# 4. METRIC CALCULATION
# ==========================================
def get_metrics(loader, model, device):
    model.eval()
    m_loss, m_acc, m_prec, m_rec, m_f1, m_iou, m_dice = [], [], [], [], [], [], []
    criterion_bce, criterion_dice = nn.BCELoss(), DiceLoss()
    
    with torch.no_grad():
        for imgs, msks in loader:
            imgs, msks = imgs.to(device), msks.to(device)
            outputs = model(imgs)
            loss = criterion_bce(outputs, msks) + criterion_dice(outputs, msks)
            preds = (outputs > 0.5).float()
            
            y_true = msks.cpu().numpy().flatten()
            y_pred = preds.cpu().numpy().flatten()
            
            m_loss.append(loss.item())
            m_acc.append((preds == msks).sum().item() / torch.numel(msks))
            m_prec.append(precision_score(y_true, y_pred, zero_division=0))
            m_rec.append(recall_score(y_true, y_pred, zero_division=0))
            m_f1.append(f1_score(y_true, y_pred, zero_division=0))
            
            inter = (preds * msks).sum().item()
            union = (preds + msks).clamp(0, 1).sum().item()
            m_iou.append((inter + 1e-6) / (union + 1e-6))
            m_dice.append((2 * inter + 1e-6) / (preds.sum().item() + msks.sum().item() + 1e-6))
            
    return np.mean(m_loss), np.mean(m_acc), np.mean(m_prec), np.mean(m_rec), np.mean(m_f1), np.mean(m_iou), np.mean(m_dice)

# ==========================================
# 5. MAIN K-FOLD PROCESS
# ==========================================
def train_with_kfold():
    image_dir = r"D:\Biose\After_Rework\1000_Dataset\image"
    mask_dir = r"D:\Biose\After_Rework\1000_Dataset\label"
    
    batch_size, lr, epochs, k_folds = 4, 1e-4, 100, 5 
    device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = SegmentationDataset(image_dir, mask_dir, augment=True)
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    final_csv_data = []

    for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
        print(f"\n--- FOLD {fold + 1} ---")
        train_loader = DataLoader(Subset(dataset, train_ids), batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(Subset(dataset, val_ids), batch_size=1, shuffle=False)
        
        model = UNet().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion_bce, criterion_dice = nn.BCELoss(), DiceLoss()
        early_stopper = EarlyStopping(patience=10, min_delta=0.001)

        for epoch in range(epochs):
            model.train()
            for imgs, msks in train_loader:
                imgs, msks = imgs.to(device), msks.to(device)
                optimizer.zero_grad()
                out = model(imgs)
                loss = criterion_bce(out, msks) + criterion_dice(out, msks)
                loss.backward(); optimizer.step()

            # Early Stopping Check
            v_loss_only, _, _, _, _, _, _ = get_metrics(val_loader, model, device)
            early_stopper(v_loss_only)
            
            if early_stopper.early_stop:
                print(f"Early Stopping at epoch {epoch+1}")
                break

        # Collect metrics for CSV
        t_metrics = get_metrics(train_loader, model, device)
        v_metrics = get_metrics(val_loader, model, device)
        
        final_csv_data.append({
            'fold': fold + 1,
            'train_loss': t_metrics[0], 'train_accu': t_metrics[1], 'train_dice': t_metrics[6],
            'val_loss': v_metrics[0], 'val_accu': v_metrics[1], 'val_dice': v_metrics[6]
        })
        print(f"Fold {fold+1} Finished. Val Dice: {v_metrics[6]:.4f}")

        # === SAVE BLOCK ADDED HERE ===
        # This saves the model for each fold separately
        torch.save(model.state_dict(), f"unet_fold_{fold+1}.pth")

    pd.DataFrame(final_csv_data).to_csv("comprehensive_metrics.csv", index=False)
    print("\nCSV Saved and Models saved for all folds. Process Complete.")

train_with_kfold()

Dataset Initialized: Found 1000 images/masks.

--- FOLD 1 ---
Early Stopping at epoch 44
Fold 1 Finished. Val Dice: 0.7512

--- FOLD 2 ---
Early Stopping at epoch 42
Fold 2 Finished. Val Dice: 0.7008

--- FOLD 3 ---
Early Stopping at epoch 50
Fold 3 Finished. Val Dice: 0.7750

--- FOLD 4 ---
Early Stopping at epoch 42
Fold 4 Finished. Val Dice: 0.7127

--- FOLD 5 ---
Early Stopping at epoch 42
Fold 5 Finished. Val Dice: 0.6474

CSV Saved and Models saved for all folds. Process Complete.


In [4]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import KFold

def fix_metrics_csv():
    # 1. Setup (Same paths and parameters as before)
    image_dir = r"D:\Biose\After_Rework\1000_Dataset\image"
    mask_dir = r"D:\Biose\After_Rework\1000_Dataset\label"
    k_folds = 5
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Use augment=False for metric evaluation to get clean results
    dataset = SegmentationDataset(image_dir, mask_dir, augment=False)
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    full_metrics_data = []

    print("Starting metric extraction from saved models...")

    for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
        print(f"Processing Fold {fold + 1}...")
        
        # Recreate the exact same data split
        train_loader = DataLoader(Subset(dataset, train_ids), batch_size=4, shuffle=False)
        val_loader = DataLoader(Subset(dataset, val_ids), batch_size=1, shuffle=False)
        
        # Load the saved weight for this specific fold
        model = UNet().to(device)
        model_path = f"unet_fold_{fold+1}.pth"
        
        if os.path.exists(model_path):
            model.load_state_dict(torch.load(model_path))
            
            # Use your existing get_metrics function to get all 7 values
            t_m = get_metrics(train_loader, model, device)
            v_m = get_metrics(val_loader, model, device)
            
            # Build the complete row
            row = {
                'fold': fold + 1,
                'train_loss': t_m[0], 'train_accu': t_m[1], 'train_prec': t_m[2], 
                'train_reca': t_m[3], 'train_f1': t_m[4], 'train_iou': t_m[5], 'train_dice': t_m[6],
                'val_loss': v_m[0], 'val_accu': v_m[1], 'val_prec': v_m[2], 
                'val_reca': v_m[3], 'val_f1': v_m[4], 'val_iou': v_m[5], 'val_dice': v_m[6]
            }
            full_metrics_data.append(row)
        else:
            print(f"Warning: {model_path} not found. Skipping fold.")

    # Save to a new CSV file
    df = pd.DataFrame(full_metrics_data)
    df.to_csv("comprehensive_metrics_fixed.csv", index=False)
    
    print("\nSuccess! All metrics calculated.")
    print("New file created: comprehensive_metrics_fixed.csv")
    return df

# Execute the fix
fix_metrics_csv()

Dataset Initialized: Found 1000 images/masks.
Starting metric extraction from saved models...
Processing Fold 1...
Processing Fold 2...
Processing Fold 3...
Processing Fold 4...
Processing Fold 5...

Success! All metrics calculated.
New file created: comprehensive_metrics_fixed.csv


,fold,train_loss,train_accu,train_prec,train_reca,train_f1,train_iou,train_dice,val_loss,val_accu,val_prec,val_reca,val_f1,val_iou,val_dice
0,1,0.183447,0.975442,0.923736,0.915107,0.917354,0.850646,0.917354,0.565339,0.927594,0.781042,0.778487,0.740562,0.636440,0.740562
1,2,0.200033,0.973330,0.915947,0.908152,0.909337,0.837308,0.909337,0.648108,0.923496,0.730582,0.752003,0.693899,0.596153,0.693899
2,3,0.213390,0.971580,0.933305,0.883114,0.904234,0.829727,0.904234,0.497799,0.935218,0.813327,0.827674,0.784872,0.689393,0.784872
3,4,0.246606,0.965624,0.950433,0.836256,0.885913,0.800554,0.885913,0.558278,0.933660,0.790931,0.704264,0.701690,0.601738,0.701690
4,5,0.311452,0.955682,0.793285,0.938883,0.853967,0.753141,0.853967,0.785334,0.895089,0.620672,0.807973,0.643930,0.527279,0.643930
